# Kratos MCP Server — interactive sessions: load stepping

Most of this server's simulations run as **batch jobs** (`run_simulation`): you hand over a case and get a
result. This notebook drives the **other** execution model — an *interactive session*: the server keeps a live
Kratos `Model` and an initialized `AnalysisStage` in a long-lived worker process, and we advance it **a few steps
at a time**, look at the fields, nodes and elements in between, change the state, and continue.

It uses a 5 m × 0.5 m cantilever whose tip load is ramped with time (`"1.0e6*t"`), and covers:

- **`session_start` / `session_advance`** — stepping with the same sequence `AnalysisStage.Run()` uses;
- **`session_field_summary`, `session_inspect_node`, `session_inspect_element`, `session_model_info`** — querying the live state;
- **`session_set_nodal_values`** — steering the model mid-run (pinning a DOF);
- **`session_exec_python`** — the opt-in escape hatch into Kratos' Python layer;
- a check that the **stepped result equals a batch `run_simulation`** of the same files.

Pairs with the `kratos://examples/load-stepping-cantilever` resource, the `interactive_session` prompt and the
`docs/tutorials/interactive-session.md` tutorial. Every output is from a live run against a Kratos 10.4 build
(pip wheels or compiled) with StructuralMechanicsApplication and LinearSolversApplication.

In [1]:
import asyncio
import base64
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path
from contextlib import AsyncExitStack

from IPython.display import Image as IPImage, Markdown, display

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def _preview(value, limit=4000):
    text = value if isinstance(value, str) else json.dumps(value, indent=2, default=str)
    if len(text) > limit:
        text = text[: limit] + f"\n... [truncated, {len(text)} chars total]"
    print(text)


async def call(tool_name, **kwargs):
    # Call an MCP tool, print its JSON result, display any inline images,
    # and return the parsed value for use in later cells.
    result = await session.call_tool(tool_name, kwargs)
    value = None
    for block in result.content:
        if block.type == "text":
            try:
                value = json.loads(block.text)
            except json.JSONDecodeError:
                value = block.text
        elif block.type == "image":
            display(IPImage(data=base64.b64decode(block.data), format=block.mimeType.split("/")[-1]))
    _preview(value)
    return value


async def read_resource(uri):
    result = await session.read_resource(uri)
    return result.contents[0].text


async def get_prompt(name, **kwargs):
    result = await session.get_prompt(name, kwargs)
    return result.messages[0].content.text

## Connect to the server

The server's stdio pipes are managed by anyio context managers whose cancel scopes are tied to the asyncio Task that opened them. Every Jupyter cell runs its top-level `await` in a *new* Task, so we run the whole connect → disconnect lifecycle inside one persistent background task kept alive on an `asyncio.Event`, and only *signal* it to close. We start this demo server with `KRATOS_MCP_ALLOW_EXEC=1` so the opt-in `session_exec_python` tool works.

In [2]:
session = None
_stop_event = asyncio.Event()
_ready_event = asyncio.Event()


async def _server_lifecycle():
    global session
    async with AsyncExitStack() as stack:
        # session_exec_python is opt-in: enable it for this demo server only
        params = StdioServerParameters(
            command=sys.executable, args=["-m", "kratos_mcp.server"],
            env={**os.environ, "KRATOS_MCP_ALLOW_EXEC": "1"})
        read, write = await stack.enter_async_context(stdio_client(params))
        session = await stack.enter_async_context(ClientSession(read, write))
        await session.initialize()
        _ready_event.set()
        await _stop_event.wait()  # keeps this task (and the session) alive


_server_task = asyncio.create_task(_server_lifecycle())
await _ready_event.wait()

tools = await session.list_tools()
resources = await session.list_resources()
prompts = await session.list_prompts()
print(f"{len(tools.tools)} tools, {len(resources.resources)} resources, "
      f"{len(prompts.prompts)} prompts")
print("session tools:", sorted(t.name for t in tools.tools if t.name.startswith("session_")))
print("prompts:", [p.name for p in prompts.prompts])

64 tools, 14 resources, 6 prompts
session tools: ['session_advance', 'session_close', 'session_exec_python', 'session_field_summary', 'session_get_parameters', 'session_inspect_element', 'session_inspect_node', 'session_list', 'session_model_info', 'session_set_nodal_values', 'session_start', 'session_status']
prompts: ['setup_structural_analysis', 'setup_thermal_analysis', 'setup_fluid_analysis', 'debug_failed_simulation', 'postprocess_results', 'interactive_session']


## 1. The case

The worked-example resource carries the mesh, `ProjectParameters.json`, `Materials.json` and the verified result.
The mesh listing is long, so we print the introduction and the result sections only.

In [3]:
text = await read_resource("kratos://examples/load-stepping-cantilever")
print(text.split("## mesh.mdpa")[0])
print(text[text.index("## Verified result"):])

# Worked example: load-stepping cantilever (interactive session)

A 5 m x 0.5 m plane-strain steel cantilever (105 nodes, 80 `SmallDisplacementElement2D4N`
quads, E = 210 GPa, nu = 0.3) fixed on `Structure.left`, with a tip line load on
`Structure.right` that is RAMPED with time: `modulus = "1.0e6*t"` N/m, pointing
down. `time_step` is 0.125 (exactly representable) so the 8 steps land on
t = 0.125, 0.25, ..., 1.0 with no floating-point drift.

Unlike the other examples this one is meant to be driven STEP BY STEP with the
`session_*` tools instead of run to completion with `run_simulation`: open it
with `session_start`, advance a couple of steps, inspect the displacement field
and the tip node, advance the rest, and close. Because the problem is linear
the stepped result must equal a batch run of the same files.

Real files on disk (src/kratos_mcp/examples/load_stepping_cantilever/); the
mesh is a 20x4 rectangle from mdpa_create_structured_mesh.


## Verified result

Stepping the case w

## 2. Stage the case and validate it

Copy the real example files to a working directory (a session can also `isolate` them itself, which we do below).

In [4]:
import kratos_mcp  # only to locate the bundled example on disk
workdir = Path(tempfile.mkdtemp(prefix="kratos-mcp-session-"))
example_dir = Path(kratos_mcp.__file__).parent / "examples" / "load_stepping_cantilever"
for f in ("mesh.mdpa", "ProjectParameters.json", "Materials.json"):
    shutil.copy(example_dir / f, workdir / f)
print("case directory:", workdir)
await call("validate_case", case_dir=str(workdir))

case directory: /tmp/kratos-mcp-session-zvsqoetx


{
  "valid": true,
  "issues": [],
  "warnings": [],
  "deep_validated": true
}


{'valid': True, 'issues': [], 'warnings': [], 'deep_validated': True}

## 3. Open a session

`session_start` loads the case, builds the `AnalysisStage` and runs `Initialize()` — nothing is solved yet.
`isolate=True` makes the session work on a private copy, so the files above stay pristine (we reuse them for the batch comparison at the end).

In [5]:
start = await call("session_start", case_dir=str(workdir), isolate=True)
sid = start["session_id"]

{
  "session_id": "20261010-110502-379dbd",
  "state": "ready",
  "case_dir": "/root/.kratos-mcp/sessions/20261010-110502-379dbd/execution",
  "log_file": "/root/.kratos-mcp/sessions/20261010-110502-379dbd/session.log",
  "analysis_class": "KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis:StructuralMechanicsAnalysis",
  "time": 0.0,
  "end_time": 1.0,
  "step": 0,
  "steps_taken": 0,
  "finished": false,
  "finalized": false,
  "model_part": "Structure",
  "nodes": 105,
  "elements": 80,
  "conditions": 48,
  "model_parts": [
    "Structure",
    "Structure.bottom",
    "Structure.top",
    "Structure.left",
    "Structure.right",
    "Structure.domain"
  ]
}


What will actually be solved, after Kratos applied its defaults? Query the live parameters with a JSON Pointer:

In [6]:
await call("session_get_parameters", session_id=sid, path="/solver_settings/time_stepping")
await call("session_get_parameters", session_id=sid, path="/processes/loads_process_list/0/Parameters/modulus")

{
  "path": "/solver_settings/time_stepping",
  "parameters": {
    "time_step": 0.125
  }
}
{
  "path": "/processes/loads_process_list/0/Parameters/modulus",
  "parameters": "1.0e6*t"
}


{'path': '/processes/loads_process_list/0/Parameters/modulus',
 'parameters': '1.0e6*t'}

## 4. Advance two steps and look around

The load is `1.0e6 * t` with `time_step = 0.125`, so two steps bring us to t = 0.25. The per-step `history` records
whether each solve converged.

In [7]:
adv = await call("session_advance", session_id=sid, steps=2)

{
  "advanced": 2,
  "time": 0.25,
  "end_time": 1.0,
  "finished": false,
  "stop_reason": "steps_completed",
  "history": [
    {
      "step": 1,
      "time": 0.125,
      "converged": true,
      "wall_seconds": 0.0056
    },
    {
      "step": 2,
      "time": 0.25,
      "converged": true,
      "wall_seconds": 0.0029
    }
  ]
}


In [8]:
# where is the displacement largest?
await call("session_field_summary", session_id=sid, variable="DISPLACEMENT")

{
  "count": 105,
  "kind": "vector",
  "min": 0.0,
  "max": 0.0019721555749972535,
  "mean": 0.0007543667832427863,
  "argmax": {
    "id": 21,
    "value": [
      -0.00014680660744078618,
      -0.001966683866818578,
      0.0
    ],
    "magnitude": 0.0019721555749972535,
    "coordinates": [
      4.99985319339256,
      -0.001966683866818578,
      0.0
    ]
  },
  "argmin": {
    "id": 1,
    "value": [
      0.0,
      0.0,
      0.0
    ],
    "magnitude": 0.0,
    "coordinates": [
      0.0,
      0.0,
      0.0
    ]
  },
  "component_max": [
    0.0001468066073530886,
    0.0,
    0.0
  ],
  "component_min": [
    -0.00014680660744078618,
    -0.001966683866818578,
    0.0
  ],
  "variable": "DISPLACEMENT",
  "model_part": "Structure",
  "location": "nodes",
  "time": 0.25
}


{'count': 105,
 'kind': 'vector',
 'min': 0.0,
 'max': 0.0019721555749972535,
 'mean': 0.0007543667832427863,
 'argmax': {'id': 21,
  'value': [-0.00014680660744078618, -0.001966683866818578, 0.0],
  'magnitude': 0.0019721555749972535,
  'coordinates': [4.99985319339256, -0.001966683866818578, 0.0]},
 'argmin': {'id': 1,
  'value': [0.0, 0.0, 0.0],
  'magnitude': 0.0,
  'coordinates': [0.0, 0.0, 0.0]},
 'component_max': [0.0001468066073530886, 0.0, 0.0],
 'component_min': [-0.00014680660744078618, -0.001966683866818578, 0.0],
 'variable': 'DISPLACEMENT',
 'model_part': 'Structure',
 'location': 'nodes',
 'time': 0.25}

In [9]:
# the loaded tip node, and the stress field evaluated at the Gauss points of every element
await call("session_inspect_node", session_id=sid, node_id=105, variables=["DISPLACEMENT", "REACTION"])
await call("session_field_summary", session_id=sid, variable="VON_MISES_STRESS", location="elements")

{
  "id": 105,
  "coordinates": [
    5.000146806607353,
    0.498033316133359,
    0.0
  ],
  "initial_coordinates": [
    5.0,
    0.5,
    0.0
  ],
  "values": {
    "DISPLACEMENT": [
      0.0001468066073530886,
      -0.0019666838666410158,
      0.0
    ],
    "REACTION": [
      -1.284934114664793e-08,
      -0.002435395599604817,
      0.0
    ]
  },
  "fixed": {
    "DISPLACEMENT": [
      false,
      false,
      false
    ],
    "REACTION": [
      false,
      false,
      false
    ]
  },
  "time": 0.25
}
{
  "count": 80,
  "kind": "scalar",
  "min": 488498.1815172705,
  "max": 12091846.8844095,
  "mean": 4736814.2778319735,
  "argmax": {
    "id": 1,
    "value": 12091846.8844095,
    "magnitude": 12091846.8844095
  },
  "argmin": {
    "id": 80,
    "value": 488498.1815172705,
    "magnitude": 488498.1815172705
  },
  "variable": "VON_MISES_STRESS",
  "model_part": "Structure",
  "location": "elements",
  "time": 0.25
}


{'count': 80,
 'kind': 'scalar',
 'min': 488498.1815172705,
 'max': 12091846.8844095,
 'mean': 4736814.2778319735,
 'argmax': {'id': 1, 'value': 12091846.8844095, 'magnitude': 12091846.8844095},
 'argmin': {'id': 80,
  'value': 488498.1815172705,
  'magnitude': 488498.1815172705},
 'variable': 'VON_MISES_STRESS',
 'model_part': 'Structure',
 'location': 'elements',
 'time': 0.25}

In [10]:
# the clamped edge carries the reaction; sub model parts are addressable by name
await call("session_field_summary", session_id=sid, variable="REACTION", model_part="Structure.left")
await call("session_model_info", session_id=sid, depth=1)

{
  "count": 5,
  "kind": "vector",
  "min": 228682.43217091734,
  "max": 948437.9153135153,
  "mean": 746113.952472115,
  "argmax": {
    "id": 1,
    "value": [
      863484.6034693632,
      392337.6336468421,
      0.0
    ],
    "magnitude": 948437.9153135153,
    "coordinates": [
      0.0,
      0.0,
      0.0
    ]
  },
  "argmin": {
    "id": 43,
    "value": [
      2.2246240405365825e-08,
      -228682.43217091734,
      0.0
    ],
    "magnitude": 228682.43217091734,
    "coordinates": [
      0.0,
      0.25,
      0.0
    ]
  },
  "component_max": [
    863484.6034693632,
    392337.6336468421,
    0.0
  ],
  "component_min": [
    -863484.6034693331,
    -228682.43217091734,
    0.0
  ],
  "variable": "REACTION",
  "model_part": "left",
  "location": "nodes",
  "time": 0.25
}
{
  "name": "Structure",
  "nodes": 105,
  "elements": 80,
  "conditions": 48,
  "sub_model_parts": [
    {
      "name": "bottom",
      "nodes": 21,
      "elements": 0,
      "conditions": 20
   

{'name': 'Structure',
 'nodes': 105,
 'elements': 80,
 'conditions': 48,
 'sub_model_parts': [{'name': 'bottom',
   'nodes': 21,
   'elements': 0,
   'conditions': 20},
  {'name': 'top', 'nodes': 21, 'elements': 0, 'conditions': 20},
  {'name': 'left', 'nodes': 5, 'elements': 0, 'conditions': 4},
  {'name': 'right', 'nodes': 5, 'elements': 0, 'conditions': 4},
  {'name': 'domain', 'nodes': 105, 'elements': 80, 'conditions': 0}],
 'process_info': {'TIME': 0.25,
  'STEP': 2,
  'DELTA_TIME': 0.125,
  'DOMAIN_SIZE': 2},
 'buffer_size': 2,
 'available_model_parts': ['Structure',
  'Structure.bottom',
  'Structure.top',
  'Structure.left',
  'Structure.right',
  'Structure.domain']}

In [11]:
await call("session_inspect_element", session_id=sid, element_id=1, variables=["VON_MISES_STRESS"])

{
  "id": 1,
  "node_ids": [
    1,
    2,
    23,
    22
  ],
  "properties_id": 1,
  "active": true,
  "time": 0.25,
  "values": {
    "VON_MISES_STRESS": [
      12091846.8844095,
      11225857.929279419,
      7746935.789276378,
      7967030.486748609
    ]
  }
}


{'id': 1,
 'node_ids': [1, 2, 23, 22],
 'properties_id': 1,
 'active': True,
 'time': 0.25,
 'values': {'VON_MISES_STRESS': [12091846.8844095,
   11225857.929279419,
   7746935.789276378,
   7967030.486748609]}}

Unknown names do not hurt: the error comes back as data and the session stays usable.

In [12]:
await call("session_field_summary", session_id=sid, variable="NOT_A_VARIABLE")
(await call("session_status", session_id=sid))["state"]

{
  "error": "KeyError: \"Unknown variable 'NOT_A_VARIABLE'\""
}
{
  "session_id": "20261010-110502-379dbd",
  "state": "ready",
  "pid": 14956,
  "case_dir": "/root/.kratos-mcp/sessions/20261010-110502-379dbd/execution",
  "analysis_class": "KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis:StructuralMechanicsAnalysis",
  "age_seconds": 0.3,
  "log_file": "/root/.kratos-mcp/sessions/20261010-110502-379dbd/session.log",
  "steps_recorded": 2,
  "busy": false,
  "isolated": true,
  "original_case_dir": "/tmp/kratos-mcp-session-zvsqoetx"
}


'ready'

## 5. Step the rest one at a time

We advance one step per call and record the tip deflection and the peak von Mises stress. The problem is linear in the load ramp,
so both should grow in proportion to *t* — a sanity check we can make *while* the run is in progress.

In [13]:
async def snapshot(call_name, **kw):
    # call() prints; fetch silently for the table
    result = await session.call_tool(call_name, kw)
    return json.loads(result.content[0].text)

rows = []
for _ in range(6):
    adv = await snapshot("session_advance", session_id=sid, steps=1)
    node = await snapshot("session_inspect_node", session_id=sid, node_id=105, variables=["DISPLACEMENT"])
    vm = await snapshot("session_field_summary", session_id=sid, variable="VON_MISES_STRESS", location="elements")
    rows.append((adv["time"], node["values"]["DISPLACEMENT"][1], vm["max"], adv["history"][0]["converged"], adv["finished"]))

print(f"{'t':>6} {'uy(tip) [m]':>13} {'uy/t [m]':>11} {'max von Mises [Pa]':>19} {'conv':>5}")
for t, uy, vmax, conv, fin in rows:
    print(f"{t:6.3f} {uy:13.4e} {uy/t:11.4e} {vmax:19.4e} {str(conv):>5}")
print("finished:", rows[-1][4])
stepped_tip = rows[-1][1]

     t   uy(tip) [m]    uy/t [m]  max von Mises [Pa]  conv
 0.375   -2.9500e-03 -7.8667e-03          1.8138e+07  True
 0.500   -3.9334e-03 -7.8667e-03          2.4184e+07  True
 0.625   -4.9167e-03 -7.8667e-03          3.0230e+07  True
 0.750   -5.9001e-03 -7.8667e-03          3.6276e+07  True
 0.875   -6.8834e-03 -7.8667e-03          4.2322e+07  True
 1.000   -7.8668e-03 -7.8668e-03          4.8367e+07  True
finished: True


`uy/t` is constant (≈ −7.867e-3 m) and the stress scales the same way. Asking for more steps than remain is also safe — the loop simply stops at `end_time`:

In [14]:
await call("session_advance", session_id=sid, steps=5)

{
  "advanced": 0,
  "time": 1.0,
  "end_time": 1.0,
  "finished": true,
  "stop_reason": "end_time_reached",
  "history": []
}


{'advanced': 0,
 'time': 1.0,
 'end_time': 1.0,
 'finished': True,
 'stop_reason': 'end_time_reached',
 'history': []}

## 6. Steer the model: pin a DOF mid-run

A second, fresh session shows what a session can do that a batch job cannot. Advance one step, then **fix the tip's vertical
displacement** at its current value (a stand-in for "what if a prop is added now?"), and advance two more steps while the load keeps ramping.

In [15]:
b = await call("session_start", case_dir=str(workdir), isolate=True)
sid_b = b["session_id"]
await call("session_advance", session_id=sid_b, steps=1)
free = (await call("session_inspect_node", session_id=sid_b, node_id=105, variables=["DISPLACEMENT"]))["values"]["DISPLACEMENT"]
print("free tip displacement at t=0.125:", free)

{
  "session_id": "20261010-110503-3c8998",
  "state": "ready",
  "case_dir": "/root/.kratos-mcp/sessions/20261010-110503-3c8998/execution",
  "log_file": "/root/.kratos-mcp/sessions/20261010-110503-3c8998/session.log",
  "analysis_class": "KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis:StructuralMechanicsAnalysis",
  "time": 0.0,
  "end_time": 1.0,
  "step": 0,
  "steps_taken": 0,
  "finished": false,
  "finalized": false,
  "model_part": "Structure",
  "nodes": 105,
  "elements": 80,
  "conditions": 48,
  "model_parts": [
    "Structure",
    "Structure.bottom",
    "Structure.top",
    "Structure.left",
    "Structure.right",
    "Structure.domain"
  ]
}
{
  "advanced": 1,
  "time": 0.125,
  "end_time": 1.0,
  "finished": false,
  "stop_reason": "steps_completed",
  "history": [
    {
      "step": 1,
      "time": 0.125,
      "converged": true,
      "wall_seconds": 0.0063
    }
  ]
}
{
  "id": 105,
  "coordinates": [
    5.000073403300534,
    0.4

In [16]:
await call("session_set_nodal_values", session_id=sid_b, variable="DISPLACEMENT",
           value=free, node_ids=[105], fix=[False, True, True])
await call("session_advance", session_id=sid_b, steps=2)
pinned = await call("session_inspect_node", session_id=sid_b, node_id=105, variables=["DISPLACEMENT"])

{
  "variable": "DISPLACEMENT",
  "model_part": "Structure",
  "nodes_changed": 1,
  "value": [
    7.340330053422342e-05,
    -0.0009833418909751608,
    0.0
  ],
  "fix": [
    false,
    true,
    true
  ]
}
{
  "advanced": 2,
  "time": 0.375,
  "end_time": 1.0,
  "finished": false,
  "stop_reason": "steps_completed",
  "history": [
    {
      "step": 2,
      "time": 0.25,
      "converged": true,
      "wall_seconds": 0.0027
    },
    {
      "step": 3,
      "time": 0.375,
      "converged": true,
      "wall_seconds": 0.0021
    }
  ]
}
{
  "id": 105,
  "coordinates": [
    5.000073000450898,
    0.49901665810902485,
    0.0
  ],
  "initial_coordinates": [
    5.0,
    0.5,
    0.0
  ],
  "values": {
    "DISPLACEMENT": [
      7.300045089877863e-05,
      -0.0009833418909751608,
      0.0
    ]
  },
  "fixed": {
    "DISPLACEMENT": [
      false,
      true,
      true
    ]
  },
  "time": 0.375
}


In [17]:
print(f"tip uy at t=0.375, pinned : {pinned['values']['DISPLACEMENT'][1]:.4e} m  (held at its t=0.125 value)")
print(f"tip uy at t=0.375, unpinned: {3 * free[1]:.4e} m  (what the free run would reach)")
await call("session_list")

tip uy at t=0.375, pinned : -9.8334e-04 m  (held at its t=0.125 value)
tip uy at t=0.375, unpinned: -2.9500e-03 m  (what the free run would reach)
{
  "session_id": "20261010-110502-379dbd",
  "state": "ready",
  "case_dir": "/root/.kratos-mcp/sessions/20261010-110502-379dbd/execution",
  "analysis_class": "KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis:StructuralMechanicsAnalysis",
  "age_seconds": 1.1
}


{'session_id': '20261010-110502-379dbd',
 'state': 'ready',
 'case_dir': '/root/.kratos-mcp/sessions/20261010-110502-379dbd/execution',
 'analysis_class': 'KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis:StructuralMechanicsAnalysis',
 'age_seconds': 1.1}

## 7. Peek into the live Kratos objects

For anything the typed tools do not cover, `session_exec_python` runs Python inside the session's Kratos process. The namespace has
`KM`, `model`, `stage`, `session` and `parameters`; assign to `result` to get a value back. It is **opt-in**
(`KRATOS_MCP_ALLOW_EXEC=1`, which this notebook's server sets) because it executes arbitrary code as your user.

In [18]:
await call("session_exec_python", session_id=sid_b, code='''
mp = model["Structure"]
node = mp.GetNode(75)
print("hello from inside Kratos")
result = {
    "node75_initial_x": node.X0,
    "time": stage.time,
    "step": mp.ProcessInfo[KM.STEP],
    "solver": type(stage._GetSolver()).__name__,
}
''')

{
  "result": {
    "node75_initial_x": 2.75,
    "time": 0.375,
    "step": 3,
    "solver": "StaticMechanicalSolver"
  },
  "stdout": "hello from inside Kratos\n"
}


{'result': {'node75_initial_x': 2.75,
  'time': 0.375,
  'step': 3,
  'solver': 'StaticMechanicalSolver'},
 'stdout': 'hello from inside Kratos\n'}

## 8. Close the sessions

`session_close` runs `AnalysisStage.Finalize()` (so output processes flush their files) and terminates the worker.

In [19]:
await call("session_close", session_id=sid_b)
closed = await call("session_close", session_id=sid)
await call("session_status", session_id=sid)

{
  "session_id": "20261010-110503-3c8998",
  "state": "closed",
  "finalized": true
}


{
  "session_id": "20261010-110502-379dbd",
  "state": "closed",
  "finalized": true
}
{
  "session_id": "20261010-110502-379dbd",
  "state": "closed",
  "pid": 14956,
  "case_dir": "/root/.kratos-mcp/sessions/20261010-110502-379dbd/execution",
  "analysis_class": "KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis:StructuralMechanicsAnalysis",
  "age_seconds": 1.3,
  "log_file": "/root/.kratos-mcp/sessions/20261010-110502-379dbd/session.log",
  "steps_recorded": 8,
  "returncode": 0,
  "isolated": true,
  "original_case_dir": "/tmp/kratos-mcp-session-zvsqoetx"
}


{'session_id': '20261010-110502-379dbd',
 'state': 'closed',
 'pid': 14956,
 'case_dir': '/root/.kratos-mcp/sessions/20261010-110502-379dbd/execution',
 'analysis_class': 'KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis:StructuralMechanicsAnalysis',
 'age_seconds': 1.3,
 'log_file': '/root/.kratos-mcp/sessions/20261010-110502-379dbd/session.log',
 'steps_recorded': 8,
 'returncode': 0,
 'isolated': True,
 'original_case_dir': '/tmp/kratos-mcp-session-zvsqoetx'}

## 9. Does stepping change the physics? Compare with a batch run

Run the very same case files as an ordinary background job and compare the final tip deflection.

In [20]:
batch_dir = Path(tempfile.mkdtemp(prefix="kratos-mcp-session-batch-"))
for f in ("mesh.mdpa", "ProjectParameters.json", "Materials.json"):
    shutil.copy(workdir / f, batch_dir / f)
run = await call("run_simulation", case_dir=str(batch_dir), wait_seconds=120)

{
  "job_id": "20261010-110503-004366",
  "case_dir": "/tmp/kratos-mcp-session-batch-wa77khsx",
  "parameters_file": "ProjectParameters.json",
  "state": "succeeded",
  "pid": 14970,
  "returncode": 0,
  "created_at": 1791630303.8430264,
  "started_at": 1791630303.8430269,
  "finished_at": 1791630305.8469326,
  "analysis_type": null,
  "extra": {
    "original_case_dir": "/tmp/kratos-mcp-session-batch-wa77khsx",
    "execution_case_dir": "/tmp/kratos-mcp-session-batch-wa77khsx",
    "snapshot_dir": null,
    "analysis_class": null,
    "mpi": null,
    "omp_threads": null,
    "manifest": "/root/.kratos-mcp/jobs/20261010-110503-004366/manifest.json"
  },
  "elapsed_seconds": 2.0
}


In [21]:
import meshio, numpy as np
vtk = sorted((batch_dir / "vtk_output").glob("*.vtk"))[-1]
m = meshio.read(str(vtk))
batch_tip = float(np.asarray(m.point_data["DISPLACEMENT"])[104][1])   # node 105 is row 104
print(f"final VTK file          : {vtk.name}")
print(f"batch   tip uy (t=1.0)  : {batch_tip:.6e} m")
print(f"stepped tip uy (t=1.0)  : {stepped_tip:.6e} m")
print(f"relative difference     : {abs(batch_tip - stepped_tip) / abs(batch_tip):.2e}")

final VTK file          : Structure_0_8.vtk
batch   tip uy (t=1.0)  : -7.866751e-03 m
stepped tip uy (t=1.0)  : -7.866752e-03 m
relative difference     : 4.30e-08


The two agree to numerical precision: a session changes *when you can look and what you can change*, not what Kratos computes — the worker
replays the identical `AnalysisStage` step sequence that `Run()` does.

For the guided version of this workflow, ask the server for the prompt:

In [22]:
print(await get_prompt("interactive_session", case_dir=str(workdir)))

Drive the Kratos case in '/tmp/kratos-mcp-session-zvsqoetx' interactively: step the case forward, checking the response after every few steps

Use a session (a live Model + AnalysisStage kept in memory) instead of
run_simulation when you want to look at the state between steps:
1. validate_case('/tmp/kratos-mcp-session-zvsqoetx') — fix problems first; sessions reject
   multi-stage and MPI cases.
2. session_start(case_dir='/tmp/kratos-mcp-session-zvsqoetx', isolate=true) — isolate keeps the
   original files untouched. Note the session_id, end_time and model parts.
3. session_get_parameters(session_id, path='/solver_settings') — confirm what
   is actually solved (time step, solver, processes).
4. Loop: session_advance(steps=N) -> session_field_summary(variable=...) ->
   session_inspect_node / session_inspect_element at the hot spot. Check each
   chunk is converged and the response is physically plausible before going on.
5. Optionally change the live state with session_set_nodal_val

## Cleanup

Close the client session (which stops the server subprocess, and with it any session still open).

In [23]:
_stop_event.set()      # wake the lifecycle task up ...
await _server_task     # ... and let it exit its own AsyncExitStack from the task that opened it
print("session closed, server subprocess terminated")
print("case directory left on disk:", workdir)

session closed, server subprocess terminated
case directory left on disk: /tmp/kratos-mcp-session-zvsqoetx
